# Add: Transformer with Attention Layer

In [2]:
data = [
  "She composes songs and practices piano daily.",
  "He reads books and explores the nearby caves.",
  "He reads novel and climbs mountains every weekend.",
  "She composes songs and writes novels.",
  "He reads newspaper and solves complex puzzles.",
  "She composes music and organizes exhibitions regularly.",
  "He reads books and builds small wooden models.",
  "He reads books and participates in local science fairs.",
  "She composes songs and curates art projects.",
  "She composes tunes and designs jewelry for her friends.",
  "He reads everyday and documents wildlife photography trips.",
  "She composes harmonies and experiments with digital music.",
  "He reads novel and trains for local marathons.",
  "She composes soundtracks and collaborates with creative filmmakers.",
  "He reads newspaper and studies navigation using maps and stars.",
  "She composes rhythms and teaches music."
]

In [72]:
VOCAB_SIZE = 7000
CONTEXT_LEN = 6
EMB_DIM = 24
BATCH_SIZE = 4
EPOCHS = 100

In [73]:
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

In [74]:
_ = torch.manual_seed(123)

In [75]:
def separate_dot(text):
    text = re.sub(r'([.,!?])', r' \1 ', text)
    return text.strip()

text_data = [separate_dot(text) for text in data]

In [76]:
example = text_data[0]
print(example)

She composes songs and practices piano daily .


In [77]:
vocab = set()

for text in text_data:
    for word in text.split():
       vocab.add(word)

vocab = sorted(list(vocab))
print(len(vocab))


70


In [78]:
for i, word in enumerate(vocab):
    print(f"{i}: {word}")

0: .
1: He
2: She
3: and
4: art
5: books
6: builds
7: caves
8: climbs
9: collaborates
10: complex
11: composes
12: creative
13: curates
14: daily
15: designs
16: digital
17: documents
18: every
19: everyday
20: exhibitions
21: experiments
22: explores
23: fairs
24: filmmakers
25: for
26: friends
27: harmonies
28: her
29: in
30: jewelry
31: local
32: maps
33: marathons
34: models
35: mountains
36: music
37: navigation
38: nearby
39: newspaper
40: novel
41: novels
42: organizes
43: participates
44: photography
45: piano
46: practices
47: projects
48: puzzles
49: reads
50: regularly
51: rhythms
52: science
53: small
54: solves
55: songs
56: soundtracks
57: stars
58: studies
59: teaches
60: the
61: trains
62: trips
63: tunes
64: using
65: weekend
66: wildlife
67: with
68: wooden
69: writes


In [79]:
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for idx, word in enumerate(vocab)}

In [80]:
def text_to_token_ids(text, word_to_idx):
    token = text.split()

    return [word_to_idx[t] for t in token]

def token_to_text(token_ids, idx_to_word):
    return " ".join([idx_to_word[idx] for idx in token_ids])

In [81]:
print(example)

She composes songs and practices piano daily .


In [82]:
text_to_token_ids(example,word_to_idx)


[2, 11, 55, 3, 46, 45, 14, 0]

In [83]:
text_to_token_ids(example,word_to_idx)[: CONTEXT_LEN + 1]

[2, 11, 55, 3, 46, 45, 14]

In [84]:
text_to_token_ids(example,word_to_idx)[: CONTEXT_LEN]

[2, 11, 55, 3, 46, 45]

In [85]:
text_to_token_ids(example,word_to_idx)[: CONTEXT_LEN - 1]

[2, 11, 55, 3, 46]

In [86]:
token_to_text(text_to_token_ids(example,word_to_idx), idx_to_word)

'She composes songs and practices piano daily .'

In [87]:
class LLMDataset(Dataset):
    def __init__(self, text_data, word_to_idx, context_len):
        self.text_data = text_data
        self.word_to_idx = word_to_idx
        self.context_len = context_len


    def __len__(self):
        return len(self.text_data)

    def __getitem__(self, idx):
        tokens = torch.tensor(text_to_token_ids(self.text_data[idx], self.word_to_idx))[: self.context_len + 1]
        x = tokens[:-1]
        y = tokens[1:]
        return x,y
    
triain_dataset = LLMDataset(text_data, word_to_idx, CONTEXT_LEN)
train_dataloader = DataLoader(triain_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(len(train_dataloader))

4


In [88]:
example_input_output = next(iter(train_dataloader))
print(example_input_output)
print ("-------------------------")
example_input_output[1][0]

[tensor([[ 2, 11, 55,  3, 46, 45],
        [ 1, 49,  5,  3, 22, 60],
        [ 1, 49, 40,  3,  8, 35],
        [ 2, 11, 55,  3, 69, 41]]), tensor([[11, 55,  3, 46, 45, 14],
        [49,  5,  3, 22, 60, 38],
        [49, 40,  3,  8, 35, 18],
        [11, 55,  3, 69, 41,  0]])]
-------------------------


tensor([11, 55,  3, 46, 45, 14])

In [89]:
print ("-------------------------")
example_input_output[0][0]


-------------------------


tensor([ 2, 11, 55,  3, 46, 45])

In [90]:
print ("-------------------------")
example_input_output[0][1]

-------------------------


tensor([ 1, 49,  5,  3, 22, 60])

In [91]:
class Attantion(nn.Module):
    def __init__(self,d_in,d_out,context_length):
        super().__init__()
        self.d_out = d_out
        self.W_q = nn.Linear(d_in, d_out,bias=False)
        self.W_k = nn.Linear(d_in, d_out,bias=False)
        self.W_v = nn.Linear(d_in, d_out,bias=False)

    def forward(self, x, return_weights=False):
        _,num_tokens,_ = x.shape
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        attn_scores = Q @ K.transpose(1, 2) 
        mask_bool = self.mask[:num_tokens, :num_tokens].bool()
        attn_scores = attn_scores.masked_fill(~mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores/(self.d_out **0.5), dim=-1)
        context_vec = attn_weights @ V

        if return_weights:
            return context_vec, attn_weights

        return context_vec

In [92]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = Attantion(
            d_in = EMB_DIM,
            d_out = EMB_DIM,
            context_length = CONTEXT_LEN)
      

    def forward(self, x):
        shortcut = x
        x = self.attention(x)
        x = x + shortcut  
        return x

In [93]:
class GPTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(VOCAB_SIZE, EMB_DIM)
        self.position_embedding = nn.Embedding(CONTEXT_LEN, EMB_DIM)
        self.transformer_block1 = TransformerBlock()
        self.transformer_block2 = TransformerBlock()
        self.transformer_block3 = TransformerBlock()
        self.transformer_block4 = TransformerBlock()
        self.transformer_block5 = TransformerBlock()
        self.transformer_block6 = TransformerBlock()
        self.output_layer = nn.Linear(EMB_DIM, VOCAB_SIZE, bias=False)



    def forward(self, in_idx):
       _,seg_len = in_idx.shape
       token_emb = self.token_embedding(in_idx)
       pos_enbeds = self.position_embedding(torch.arange(seg_len)) 
       x = token_emb + pos_enbeds
       x = self.transformer_block1(x)
       x = self.transformer_block2(x)
       x = self.transformer_block3(x)
       x = self.transformer_block4(x)
       x = self.transformer_block5(x)
       x = self.transformer_block6(x) 
       
       logits = self.output_layer(x)
       return logits

In [94]:
model = GPTModel()
print(model)

GPTModel(
  (token_embedding): Embedding(7000, 24)
  (position_embedding): Embedding(6, 24)
  (transformer_block1): TransformerBlock(
    (attention): Attantion(
      (W_q): Linear(in_features=24, out_features=24, bias=False)
      (W_k): Linear(in_features=24, out_features=24, bias=False)
      (W_v): Linear(in_features=24, out_features=24, bias=False)
    )
  )
  (transformer_block2): TransformerBlock(
    (attention): Attantion(
      (W_q): Linear(in_features=24, out_features=24, bias=False)
      (W_k): Linear(in_features=24, out_features=24, bias=False)
      (W_v): Linear(in_features=24, out_features=24, bias=False)
    )
  )
  (transformer_block3): TransformerBlock(
    (attention): Attantion(
      (W_q): Linear(in_features=24, out_features=24, bias=False)
      (W_k): Linear(in_features=24, out_features=24, bias=False)
      (W_v): Linear(in_features=24, out_features=24, bias=False)
    )
  )
  (transformer_block4): TransformerBlock(
    (attention): Attantion(
      (W_q): 

In [95]:
model = GPTModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()

In [96]:
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {trainable_parameters:,}")

Total trainable parameters: 346,512


In [97]:
param_count = defaultdict(int)
for name, param in model.named_parameters():
   top_level_module = name.split('.')[0]
   param_count[top_level_module] += param.numel()

for key, value in param_count.items():
    print(f"{key:20s}: {value:,}")   

token_embedding     : 168,000
position_embedding  : 144
transformer_block1  : 1,728
transformer_block2  : 1,728
transformer_block3  : 1,728
transformer_block4  : 1,728
transformer_block5  : 1,728
transformer_block6  : 1,728
output_layer        : 168,000


In [111]:
def print_module_parameter(module, indent=0):
    for name, child in module.named_children():
        print(' ' * indent + f"[{name}]")
        
        for p_name, p in child.named_parameters(recurse = False):
             print(" " * (indent + 2) + f"{p_name}: {p.numel():,}")
        print_module_parameter(child, indent + 4)
                  

In [112]:
print_module_parameter(model.transformer_block1)

[attention]
    [W_q]
      weight: 576
    [W_k]
      weight: 576
    [W_v]
      weight: 576


In [113]:
print_module_parameter(model.transformer_block2)

[attention]
    [W_q]
      weight: 576
    [W_k]
      weight: 576
    [W_v]
      weight: 576


In [114]:
print_module_parameter(model.transformer_block6)

[attention]
    [W_q]
      weight: 576
    [W_k]
      weight: 576
    [W_v]
      weight: 576


In [118]:
def train(model, dataloader, optimizer, loss_fn):
    model.train()
    for batch, (x,y) in enumerate(dataloader):
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits.flatten(0,1), y.flatten())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        print(f"Batch {batch + 1} Loss: {loss:.7f}")

In [121]:
for epoch in range(EPOCHS):
    print("--------------------------------")
    print(f"Epoch {epoch + 1}")
    train(model, train_dataloader, optimizer, loss_fn)
    print("--------------------------------")
    print("DONE")

--------------------------------
Epoch 1


AttributeError: 'Attantion' object has no attribute 'mask'